<a href="https://colab.research.google.com/github/izzat-ai/learning-ai/blob/main/pandas/time_series.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

**Klinikaning kunlik Dataseti - AI dan olindi**

In [2]:
# Klinikaning 2026-yil boshidagi kunlik ma'lumotlar bazasi
data_ts = {
    'Sana': ['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-05', '2026-01-06', '2026-01-07', '2026-01-08', '2026-01-09'],
    'Kelgan_Bemorlar': [45, 50, 38, 55, 62, 48, 58, 65],
    'Kunlik_Tushum': [12000000, 15000000, 9500000, 18000000, 21000000, 14000000, 19500000, 23000000] # ming so'mda
}

df_ts = pd.DataFrame(data_ts)
df_ts

,Sana,Kelgan_Bemorlar,Kunlik_Tushum
0,2026-01-01,45,12000000
1,2026-01-02,50,15000000
2,2026-01-03,38,9500000
3,2026-01-05,55,18000000
4,2026-01-06,62,21000000
5,2026-01-07,48,14000000
6,2026-01-08,58,19500000
7,2026-01-09,65,23000000


In [3]:
# DF haqida umumiy ma'lumot olish
df_ts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Sana             8 non-null      object
 1   Kelgan_Bemorlar  8 non-null      int64 
 2   Kunlik_Tushum    8 non-null      int64 
dtypes: int64(2), object(1)
memory usage: 324.0+ bytes


- sana ustuni string ma'lumotlar turida ekan , uni datetime formatiga o'tkizish kerak

In [4]:
# sana ustunini datetime formatiga o'tkizish
df_ts['Sana'] = pd.to_datetime(df_ts['Sana'])
df_ts['Sana'].dtype

dtype('<M8[ns]')

In [5]:
df_ts

,Sana,Kelgan_Bemorlar,Kunlik_Tushum
0,2026-01-01,45,12000000
1,2026-01-02,50,15000000
2,2026-01-03,38,9500000
3,2026-01-05,55,18000000
4,2026-01-06,62,21000000
5,2026-01-07,48,14000000
6,2026-01-08,58,19500000
7,2026-01-09,65,23000000


In [6]:
# bemorlar kelgan sanalarni indeks qilish
df_ts = df_ts.set_index('Sana')
df_ts

,Kelgan_Bemorlar,Kunlik_Tushum
Sana,,
2026-01-01,45,12000000
2026-01-02,50,15000000
2026-01-03,38,9500000
2026-01-05,55,18000000
2026-01-06,62,21000000
2026-01-07,48,14000000
2026-01-08,58,19500000
2026-01-09,65,23000000


- sanalarga e'tibor bersak , 4-yanvar tushib qolgan

In [7]:
# 4-yanvarni indekslarga qo'shish
df_ts = df_ts.asfreq('D', fill_value=0)
df_ts

,Kelgan_Bemorlar,Kunlik_Tushum
Sana,,
2026-01-01,45,12000000
2026-01-02,50,15000000
2026-01-03,38,9500000
2026-01-04,0,0
2026-01-05,55,18000000
2026-01-06,62,21000000
2026-01-07,48,14000000
2026-01-08,58,19500000
2026-01-09,65,23000000


In [8]:
# haftalik hisobot olish
df_ts.resample('W').sum()

,Kelgan_Bemorlar,Kunlik_Tushum
Sana,,
2026-01-04,133,36500000
2026-01-11,288,95500000


- Yanvar oyining birinchi haftasida klinikaga 133 ta bemor kelgan va klinika 36.5 million so'm foyda ko'rgan
- huddi shu kabi ikkinchi haftada esa klinikaga kelgan bemorlar soni keskin oshgan va 95.5 million so'm foyda ko'rgan

In [9]:
# 3 kunlik siljidigan ma'lumotlarni aniqlash
bemorlar_3d = df_ts['Kelgan_Bemorlar'].rolling(3).mean()
bemorlar_3d

,Kelgan_Bemorlar
Sana,
2026-01-01,NaN
2026-01-02,NaN
2026-01-03,44.333333
2026-01-04,29.333333
2026-01-05,31.000000
2026-01-06,39.000000
2026-01-07,55.000000
2026-01-08,56.000000
2026-01-09,57.000000


- 1-va 2-yanvarlarda ma'lumot yetarli bo'lmagan shuning uchun ma'lumot yo'q
- 3-yanvargacha (3 kun ichida) klinikaga o'rtacha 44 ta bemor kelgan . Bu ko'rsatkich 4-yanvarga borib sezirarli pastlagan . Lekin 8-10-yanvarlar atrofi kelgan odamlar 56-57 tani tashkil qilgan

#####**UzWeather Analytics - O'zbekiston ob-havo tahlili**


- Dataset - Sun'iy intellekt tomonidan yaratiladi
- O'zbekiston Gidrometeorologiya markazi bizga 5 yillik ob-havo ma'lumotlarini bergan . Vazifamiz — mavsumiy trendlar, anomaliyalar va mintaqaviy farqlarni aniqlash


In [10]:
np.random.seed(42)

shaharlar = ['Toshkent', 'Samarqand', 'Buxoro', 'Andijon',
             'Namangan', 'Farg\'ona', 'Nukus', 'Termiz']

dates = pd.date_range(start='2020-01-01', end='2024-12-31', freq='D')

records = []
for shahar in shaharlar:
    # Har bir shahar uchun bazaviy temperatura
    baza_temp = {
        'Toshkent': 14, 'Samarqand': 15, 'Buxoro': 17,
        'Andijon': 13, 'Namangan': 13, 'Farg\'ona': 14,
        'Nukus': 13, 'Termiz': 20
    }[shahar]

    for date in dates:
        # Mavsumiy o'zgarish
        mavsum_temp = baza_temp + 18 * np.sin(
            2 * np.pi * (date.dayofyear - 80) / 365
        )
        # Kunlik tasodifiy og'ish
        kunlik = mavsum_temp + np.random.normal(0, 3)

        # Yog'ingarchilik (qish va bahorda ko'proq)
        yogin_ehtimol = 0.3 - 0.2 * np.sin(
            2 * np.pi * (date.dayofyear - 80) / 365
        )
        yogin = np.random.exponential(5) if np.random.random() \
            < yogin_ehtimol else 0

        # Shamol tezligi
        shamol = np.random.weibull(2) * 15

        records.append({
            'sana': date,
            'shahar': shahar,
            'harorat': round(kunlik, 1),
            'yogin_mm': round(yogin, 1),
            'shamol_ms': round(shamol, 1),
            'namlik': round(np.random.uniform(30, 90), 1)
        })

df = pd.DataFrame(records)

# NULL qiymatlar
null_idx = np.random.choice(len(df), 300, replace=False)
df.loc[null_idx[:100], 'harorat'] = np.nan
df.loc[null_idx[100:200], 'yogin_mm'] = np.nan
df.loc[null_idx[200:], 'namlik'] = np.nan

# Anomaliyalar (noto'g'ri o'lchovlar)
anomal_idx = np.random.choice(len(df), 50, replace=False)
df.loc[anomal_idx[:25], 'harorat'] = np.random.uniform(60, 80, 25)
df.loc[anomal_idx[25:], 'shamol_ms'] = np.random.uniform(100, 200, 25)

df.to_csv('uzweather.csv', index=False)

In [11]:
# DF ni fayldan o'qish
df = pd.read_csv("uzweather.csv")
df.head()

,sana,shahar,harorat,yogin_mm,shamol_ms,namlik
0,2020-01-01,Toshkent,-2.1,0.0,14.3,39.4
1,2020-01-02,Toshkent,-3.9,0.3,21.3,66.1
2,2020-01-03,Toshkent,-5.2,1.8,12.9,55.9
3,2020-01-04,Toshkent,-5.0,4.7,5.8,47.5
4,2020-01-05,Toshkent,-5.4,0.0,7.1,60.9


In [12]:
df.shape

(14616, 6)

In [13]:
# dataset haqida umumiy ma'lumot olish
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14616 entries, 0 to 14615
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sana       14616 non-null  object 
 1   shahar     14616 non-null  object 
 2   harorat    14516 non-null  float64
 3   yogin_mm   14516 non-null  float64
 4   shamol_ms  14616 non-null  float64
 5   namlik     14516 non-null  float64
dtypes: float64(4), object(2)
memory usage: 685.3+ KB


- sana ustunini Datetime formatiga o'tkizish kerak
- harorat , yogin_mm va namlik ustunlaridagi qiymatlar biroz kam ekan , demak NaN lar bor

In [14]:
# sana ustunini datetime formatiga o'tkizish
df['sana'] = pd.to_datetime(df['sana'])
df['sana'].dtype

dtype('<M8[ns]')

In [15]:
df.head()

,sana,shahar,harorat,yogin_mm,shamol_ms,namlik
0,2020-01-01,Toshkent,-2.1,0.0,14.3,39.4
1,2020-01-02,Toshkent,-3.9,0.3,21.3,66.1
2,2020-01-03,Toshkent,-5.2,1.8,12.9,55.9
3,2020-01-04,Toshkent,-5.0,4.7,5.8,47.5
4,2020-01-05,Toshkent,-5.4,0.0,7.1,60.9


In [16]:
# barcha ustunlar bo'yicha NaN qiymatlar nechtaligini aniqlash
df.isnull().sum()

,0
sana,0
shahar,0
harorat,100
yogin_mm,100
shamol_ms,0
namlik,100


In [17]:
# harorat ustuni haqida ma'lumot olish
df['harorat'].describe()

,harorat
count,14516.000000
mean,14.951906
std,13.494544
min,-14.000000
25%,2.700000
50%,14.900000
75%,27.125000
max,79.399433


In [18]:
# harorat ustunidagi NaN lar qanchalik tarqoqligini aniqlash
df[df['harorat'].isna()].head(20)

,sana,shahar,harorat,yogin_mm,shamol_ms,namlik
238,2020-08-26,Toshkent,NaN,23.6,7.7,86.6
514,2021-05-29,Toshkent,NaN,0.0,23.5,66.8
648,2021-10-10,Toshkent,NaN,0.0,23.9,71.7
1078,2022-12-14,Toshkent,NaN,9.4,10.1,77.6
1082,2022-12-18,Toshkent,NaN,0.0,7.2,47.5
1360,2023-09-22,Toshkent,NaN,0.0,15.2,69.0
1370,2023-10-02,Toshkent,NaN,0.0,7.0,79.6
1454,2023-12-25,Toshkent,NaN,0.3,5.6,81.5
1860,2020-02-03,Samarqand,NaN,5.7,4.0,65.6
1889,2020-03-03,Samarqand,NaN,0.0,15.5,45.8


In [19]:
# haroratdagi NaNlarni oldingi va keyingi kunga qarab mantiqiy to'ldirish
df['harorat'] = df['harorat'].interpolate(method='linear')
df['harorat'].isnull().sum()

np.int64(0)

In [20]:
# yog'ingarchilik ustunida qancha NaN borligini aniqlash
df['yogin_mm'].isnull().sum()

np.int64(100)

In [21]:
df['yogin_mm'].describe()

,yogin_mm
count,14516.000000
mean,1.490438
std,3.536863
min,0.000000
25%,0.000000
50%,0.000000
75%,0.900000
max,50.300000


- ushbu ma'lumotlardan ko'rishimiz mumkinki - O'zbekistonda yog'ingarchilik kam bo'ladi . Eng katta qiymat 50 turganining sababi - juda kuchli yomg'ir yoki jala quyingan bitta o'sha rekord kunni anglatishi mumkin bo'ladi
- yilning ko'p kunlari yog'ingarchilik 0 ga teng bo'ladi
- yog'ingarchilik ustunidagi NaN larni to'ldirish boshqa ustunlardan farq qiladi . Agarda interpolate yoki ffill qilsak ma'lumotlarimiz buziladi . Buning uchun eng yaxshi yechim eng ko'p takrorlangan qiymat bilan to'ldirish yoki 0 bilan to'ldirish bo'ladi

In [22]:
# yog'ingarchilik ustunini NaN larini 0 bilan to'ldirish
df['yogin_mm'] = df['yogin_mm'].fillna(0)
df['yogin_mm'].isnull().sum()

np.int64(0)

In [23]:
# sana ustunini indeks qilish
df = df.set_index('sana')
df.head()

,shahar,harorat,yogin_mm,shamol_ms,namlik
sana,,,,,
2020-01-01,Toshkent,-2.1,0.0,14.3,39.4
2020-01-02,Toshkent,-3.9,0.3,21.3,66.1
2020-01-03,Toshkent,-5.2,1.8,12.9,55.9
2020-01-04,Toshkent,-5.0,4.7,5.8,47.5
2020-01-05,Toshkent,-5.4,0.0,7.1,60.9


In [24]:
# namlik ustuni ma'lumotlarini ko'rish
df['namlik'].describe()

,namlik
count,14516.000000
mean,60.052859
std,17.271863
min,30.000000
25%,45.200000
50%,60.100000
75%,74.900000
max,90.000000


In [25]:
# namlik ustunidagi NaN larni chiziqli interpolyatsiya bilan to'ldirish
df['namlik'] = df['namlik'].interpolate(method='linear')
df['namlik'].isnull().sum()

np.int64(0)

In [26]:
# barcha ustunlardagi NaN larni tekshirish
df.isnull().sum()

,0
shahar,0
harorat,0
yogin_mm,0
shamol_ms,0
namlik,0


- IQR (interquartile range) usuli orqali anomaliya qiymatlarni aniqlaymiz
- bunda 25% va 75% lik chegaralarni hisoblab , o'sha chegaradan haddan tashqari uzoqqa chiqib ketgan outlier qiymatlarni aniqlemiz

In [27]:
for u in df.select_dtypes(include=['number']).columns:
    Q1 = df[u].quantile(0.25)
    Q3 = df[u].quantile(0.75)
    IQR = Q3 - Q1

    pastki_chegara = Q1 - 1.5 * IQR
    yuqori_chegara = Q3 + 1.5 * IQR

    outliers = df[(df[u] < pastki_chegara) | (df[u] > yuqori_chegara)]
    outlier_soni = len(outliers)

    print(f" Ustun: {u.upper()}")
    print(f"   Chegaralar: [{pastki_chegara:.2f}  va  {yuqori_chegara:.2f}]")
    print(f"   Aniqlangan outlierlar soni: {outlier_soni} ta (jami dataning {outlier_soni/len(df)*100:.2f}%)")
    print("-" * 50)

 Ustun: HARORAT
   Chegaralar: [-34.05  va  63.95]
   Aniqlangan outlierlar soni: 20 ta (jami dataning 0.14%)
--------------------------------------------------
 Ustun: YOGIN_MM
   Chegaralar: [-1.35  va  2.25]
   Aniqlangan outlierlar soni: 2785 ta (jami dataning 19.05%)
--------------------------------------------------
 Ustun: SHAMOL_MS
   Chegaralar: [-6.45  va  32.35]
   Aniqlangan outlierlar soni: 169 ta (jami dataning 1.16%)
--------------------------------------------------
 Ustun: NAMLIK
   Chegaralar: [0.96  va  119.21]
   Aniqlangan outlierlar soni: 0 ta (jami dataning 0.00%)
--------------------------------------------------


- yog'ingarchilik va shamol ustunlarida outlierlarni ko'p hisoblavorgani outlierlar ko'p degani emas . Chunki O'zbekistonda aksar kunlarda yomg'ir yog'maydi va shamol juda past bo'ladi . Normal hisoblangan 15 mmlik yomg'ir yoki 12m/s lik shamolni ham outlier deb o'ylaydi

In [28]:
# harorat ustunidagi outlierlarni aniqlash va ularni NaN bilan almashtirish
Q1 = df['harorat'].quantile(0.25)
Q3 = df['harorat'].quantile(0.75)
IQR = Q3 - Q1
pastki_chegara = Q1 - 1.5 * IQR
yuqori_chegara = Q3 + 1.5 * IQR
df.loc[(df['harorat'] < pastki_chegara) | (df['harorat'] > yuqori_chegara), 'harorat'] = np.nan
df['harorat'].isnull().sum()

np.int64(20)

In [29]:
# harorat ustunidagi NaN larni oldingi va keyingi kunga qarab mantiqiy to'ldirish
df['harorat'] = df['harorat'].interpolate(method='linear')
df['harorat'].isnull().sum()

np.int64(0)

In [30]:
# har bir shahar bo'yicha yillik o'rtacha haroratni hisoblash
yillik_harorat = df.groupby('shahar')['harorat'].resample('YE').mean().unstack()
yillik_harorat

sana,2020-12-31,2021-12-31,2022-12-31,2023-12-31,2024-12-31
shahar,,,,,
Andijon,13.207377,13.073836,13.042740,13.112877,12.975273
Buxoro,17.081857,17.031507,17.037260,17.135616,16.792213
Farg'ona,14.142486,13.861644,14.133836,13.831781,14.011885
Namangan,12.761749,13.146530,13.243151,13.050685,13.001230
Nukus,13.195708,12.953562,13.109726,13.113562,13.095158
Samarqand,14.860109,14.999315,14.845890,14.972603,15.024044
Termiz,19.922678,19.796986,19.948082,20.139420,20.125273
Toshkent,14.075410,13.599589,14.167808,13.580274,13.751913


- yil davomida eng yuqori o'rtacha harorat Termiz shahrida kuzatilgan va tahminan 20 gradusni tashkil qilgan . Salqin hududlar esa Namangan va Andijon ekanini ko'rishimiz mumkin .
- 2020-yildan 2024-yilgacha shaharlarning o'rtacha harorati keskin oshib yoki tushib ketmagan

In [32]:
# Toshkent ma'lumotlarini ajratib , oylik o'rtacha trendni hisoblash
toshkent_trend = df[df['shahar'] == 'Toshkent']['harorat'].resample('ME').mean()
toshkent_trend.sample(10)

,harorat
sana,
2020-08-31,23.580645
2020-11-30,-1.373333
2023-02-28,3.432143
2024-10-31,6.416129
2023-03-31,12.000000
2020-12-31,-3.829032
2021-02-28,4.267857
2021-06-30,30.740000
2022-03-31,11.787097


- fasllar almashinuvida havo haroratlari ham o'zgarib boradi . Toshekt shahrining oylik o'rtacha haroratlarini aniqlaganimizda buni yaqqol ko'rishimiz mumkin .
- Qish oylarida Toshkentning harorati o'rtacha -2 yarim va -3.5 dan yuqori bo'lishi kuzatiladi . Yoz oylarida tabiiyki havo isiydi . Shahar esa yozda o'rtacha 28-30 gradus bo'lar ekan